In [5]:
import sqlite3
import pandas as pd
from datetime import date
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
conn = sqlite3.connect("finance.db")
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    type TEXT,
    amount REAL,
    category TEXT,
    description TEXT,
    date TEXT
)
""")
cur.execute("""
CREATE TABLE IF NOT EXISTS budgets (
    month TEXT PRIMARY KEY,
    amount REAL
)
""")
conn.commit()
current_month = date.today().strftime("%Y-%m")
out_dashboard = widgets.Output()
out_table = widgets.Output()
type_dropdown = widgets.Dropdown(options=['Expense', 'Income'], value='Expense', description='Type:')
amount_input = widgets.FloatText(value=0.0, description='Amount (₹):')
category_dropdown = widgets.Dropdown(
    options=["Salary", "Business", "Food", "Travel", "Shopping", "Bills", "Education", "Healthcare", "Entertainment", "Other"],
    value='Food',
    description='Category:'
)
desc_input = widgets.Text(placeholder='Optional description', description='Desc:')
date_input = widgets.DatePicker(value=date.today(), description='Date:')
save_btn = widgets.Button(description='Save Transaction', button_style='primary', icon='plus')
budget_input = widgets.FloatText(value=0.0, description='Budget (₹):')
save_budget_btn = widgets.Button(description='Set Budget', button_style='success', icon='check')
delete_id_input = widgets.IntText(placeholder='Transaction ID', description='Delete ID:')
delete_btn = widgets.Button(description='Delete ID', button_style='danger', icon='trash')
export_btn = widgets.Button(description='Export to Excel', button_style='info', icon='file-excel')
def refresh_dashboard():
    with out_dashboard:
        clear_output(wait=True)
        cur.execute("""
            SELECT
            COALESCE(SUM(CASE WHEN type='Income' THEN amount ELSE 0 END), 0),
            COALESCE(SUM(CASE WHEN type='Expense' THEN amount ELSE 0 END), 0)
            FROM transactions
            WHERE substr(date, 1, 7)=?
        """, (current_month,))
        income, expenses = cur.fetchone()
        balance = income - expenses
        cur.execute("SELECT amount FROM budgets WHERE month=?", (current_month,))
        budget_res = cur.fetchone()
        budget = budget_res[0] if budget_res else 0.0
        budget_status = f"₹ {budget:,.2f}" if budget > 0 else "Not Set"
        print("=" * 60)
        print(f"   💰 PERSONAL FINANCE DASHBOARD ({current_month}) ")
        print("=" * 60)
        print(f"   Balance:  ₹ {balance:,.2f}")
        print(f"   Income:   ₹ {income:,.2f}  |  Expenses:  ₹ {expenses:,.2f}")
        print(f"   Budget:   {budget_status}")
        if budget > 0 and expenses > budget:
            print(f"   ⚠️ ALERT: You have exceeded your budget by ₹ {expenses - budget:,.2f}!")
        print("=" * 60)
        fig, ax = plt.subplots(figsize=(6, 3))
        bars = ax.bar(["Income", "Expenses"], [income, expenses], color=["#22C55E", "#EF4444"], width=0.4)
        ax.set_title("Monthly Income vs Expenses", fontweight="bold")
        ax.set_ylabel("Amount (₹)")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()
        plt.close(fig)
    with out_table:
        clear_output(wait=True)
        print("RECENT TRANSACTIONS:")
        df = pd.read_sql_query("SELECT id, type, amount, category, date, description FROM transactions ORDER BY date DESC, id DESC LIMIT 15", conn)
        if df.empty:
            print("No transactions registered yet.")
        else:
            display(df)
def save_transaction(b):
    amount = amount_input.value
    if amount <= 0:
        print("Please insert a positive amount.")
        return
    t_date = date_input.value
    if t_date and t_date > date.today():
        print("Future dates are not allowed.")
        return
    cur.execute("""
        INSERT INTO transactions (type, amount, category, description, date)
        VALUES (?, ?, ?, ?, ?)
    """, (type_dropdown.value, amount, category_dropdown.value, desc_input.value, t_date.isoformat() if t_date else date.today().isoformat()))
    conn.commit()
    amount_input.value = 0.0
    desc_input.value = ''
    refresh_dashboard()
def save_budget(b):
    amount = budget_input.value
    if amount <= 0:
         print("Please insert a positive budget.")
         return
    cur.execute("INSERT OR REPLACE INTO budgets (month, amount) VALUES (?, ?)", (current_month, amount))
    conn.commit()
    refresh_dashboard()
def delete_transaction(b):
    t_id = delete_id_input.value
    if t_id > 0:
        cur.execute("DELETE FROM transactions WHERE id=?", (t_id,))
        conn.commit()
        delete_id_input.value = 0
        refresh_dashboard()
def export_excel(b):
    df = pd.read_sql_query("SELECT * FROM transactions", conn)
    if df.empty:
        print("No transactions available to export.")
        return
    df.to_excel("transactions_export.xlsx", index=False)
    print("Successfully exported all records to 'transactions_export.xlsx'!")
save_btn.on_click(save_transaction)
save_budget_btn.on_click(save_budget)
delete_btn.on_click(delete_transaction)
export_btn.on_click(export_excel)
form_box = widgets.VBox([
    widgets.HTML("<h3>Add Transaction</h3>"),
    type_dropdown, amount_input, category_dropdown, desc_input, date_input, save_btn
])
budget_box = widgets.VBox([
    widgets.HTML("<h3>Monthly Budget</h3>"),
    budget_input, save_budget_btn
])
actions_box = widgets.VBox([
    widgets.HTML("<h3>Actions</h3>"),
    delete_id_input, delete_btn, export_btn
])
left_panel = widgets.VBox([form_box, budget_box, actions_box], layout=widgets.Layout(padding='10px', min_width='300px'))
right_panel = widgets.VBox([out_dashboard, out_table], layout=widgets.Layout(padding='10px'))
app_layout = widgets.HBox([left_panel, right_panel])
display(app_layout)
refresh_dashboard()

In [3]:
!pip install matplotlib pandas openpyxl
# Note: running python scripts that launch Tkinter GUIs will still result in a TclError in headless Colab environments.